# Chapter 02 — Data Analysis dengan NumPy & Pandas

Selamat datang di chapter 2. Di chapter 1 kamu sudah belajar fondasi Python — variabel, list, dict, loop, fungsi. Itu bekal yang cukup untuk menulis program kecil. Tapi begitu data yang kamu pegang sudah berukuran ribuan atau jutaan baris, gaya penulisan Python murni akan terasa lambat dan bertele-tele.

Di sinilah dua library muncul dan mengubah cara kita bekerja dengan data. **NumPy** adalah library yang membuat komputasi numerik di Python terasa secepat bahasa C, dengan satu struktur data utama bernama *array*. **Pandas** adalah library yang membuat data tabular (seperti spreadsheet atau tabel SQL) bisa diolah dengan Python secara natural, dengan struktur utama bernama *DataFrame*. Hampir semua hal yang akan kamu lakukan di data science — mulai dari membersihkan data, menghitung statistik, sampai menggabungkan tabel dari berbagai sumber — dilakukan dengan kedua library ini.

Chapter ini dibagi menjadi delapan section. Empat section pertama adalah dunia NumPy. Kita akan mulai dari pertanyaan sederhana: kenapa NumPy lebih cepat dari list Python? Setelah itu kita akan masuk ke *indexing*, *slicing*, dan *reshaping* array — keterampilan dasar yang kamu pakai terus-menerus. Lalu kita naik ke operasi vektor dan *broadcasting*, dua fitur yang membuat NumPy benar-benar powerful. Empat section terakhir adalah dunia Pandas. Kita mulai dari cara membuat DataFrame dan membaca file CSV (karena hampir semua data di industri datang dalam format itu), lalu cara menyaring dan memilih data dengan `.loc` dan `.iloc`. Setelah itu kita masuk ke *group by* untuk agregasi dan penanganan *missing values* — dua topik yang akan kamu temui di setiap proyek data. Section kedelapan menutup semuanya dengan mini project EDA (Exploratory Data Analysis) pada dataset yang sudah kita bangun dari awal.

Pacing chapter ini sama dengan chapter 1: satu cell = satu ide, eksperimentasi bukan demonstrasi. Kalau kamu berhenti di tengah, tidak apa-apa — buka lagi di section terakhir yang kamu ingat dan lanjut dari situ.

Sebelum kita mulai, kita import semua library yang akan dipakai. NumPy dikonvensikan sebagai `np`, Pandas sebagai `pd`, Matplotlib sebagai `plt`. Tiga huruf pendek ini adalah standar universal — kalau kamu membaca notebook orang lain di internet, kemungkinan besar mereka juga pakai alias yang sama.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"NumPy version : {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib    : {plt.__version__}")

Kalau output di atas menampilkan versi tiga library itu, berarti environment kamu sudah siap. Sekarang mari kita mulai dari pertanyaan yang sering ditanyakan pemula: kenapa sih tidak pakai list Python saja?

---

# SECTION 1: NumPy — Array Sebagai Pengganti List

Bayangkan kamu punya daftar 10 juta nilai (misalnya hasil pengukuran sensor tiap detik selama 3 bulan), dan kamu ingin menjumlahkan semuanya. Dengan list Python, kamu bisa menulis `sum(data)` — itu jalan. Tapi coba tebak berapa lama? Pada umumnya, operasi sekuensial di list Python yang sangat panjang berjalan di kisaran 0,3 sampai 1 detik, tergantung mesin. Dengan NumPy, operasi yang sama berjalan di kisaran 0,01 sampai 0,03 detik. Itu bukan dua kali lebih cepat — itu **puluhan kali lebih cepat**.

Kenapa bisa begitu? List Python menyimpan setiap elemen sebagai *object* terpisah yang punya banyak metadata (tipe, reference count, dan lain-lain), dan saat kamu melakukan `sum`, Python interpreter harus membaca tiap object satu per satu, memutuskan tipenya, lalu mengkonversi ke angka. NumPy array menyimpan semua elemen di satu blok memori yang contiguous, dengan tipe data yang seragam (semua `float64`, misalnya), dan operasi matematikanya dijalankan di kode C yang sudah dioptimasi — bukan di interpreter Python. Hasilnya: lebih sedikit overhead per elemen, dan CPU bisa memproses banyak angka dalam satu instruksi sekaligus (inilah yang disebut *SIMD* — Single Instruction Multiple Data).

Mari kita lihat sendiri perbedaannya. Di cell berikutnya, kamu akan mengetik kode benchmark sederhana. Tujuannya bukan untuk menghafal angka pasti, tapi untuk *merasakan* sendiri bahwa NumPy memang terasa berbeda di skala besar.

Coba ketik kode ini di cell di bawah. Kita akan membandingkan waktu penjumlahan 10 juta angka — sekali pakai list Python, sekali pakai NumPy array. Jangan khawatir menghafal sintaksisnya; fokuslah pada angka yang muncul di output.

In [ ]:
import time

n = 10_000_000

data_list = list(range(n))
mulai = time.time()
total_list = sum(data_list)
waktu_list = time.time() - mulai

data_np = np.arange(n)
mulai = time.time()
total_np = data_np.sum()
waktu_np = time.time() - mulai

print(f"List Python : {waktu_list:.4f} detik (hasil: {total_list:,})")
print(f"NumPy array : {waktu_np:.4f} detik (hasil: {total_np:,})")
print(f"NumPy {waktu_list/waktu_np:.1f}x lebih cepat")

Perhatikan angka "lebih cepat" di baris terakhir. Di mesin saya, NumPy sekitar 7 sampai 15 kali lebih cepat untuk penjumlahan sederhana. Untuk operasi yang lebih kompleks (misalnya perkalian matriks di neural network), keuntungannya bisa ratusan sampai ribuan kali. Jadi ketika kamu mendengar "machine learning pakai NumPy", itulah alasannya — bukan karena NumPy punya fitur ajaib, tapi karena *kecepatan* memungkinkan kita melatih model pada data yang sebenarnya.

Ada bonus lain yang tidak terlihat dari benchmark di atas: **jejak memori**. List Python untuk 10 juta integer butuh sekitar 280 MB karena setiap elemen adalah object Python lengkap. NumPy array untuk jumlah elemen yang sama butuh sekitar 80 MB karena data disimpan sebagai raw binary numbers, bukan object. Ketika kamu bekerja dengan dataset yang besar, penghematan memori ini bukan hal kecil — sering kali yang menentukan apakah laptop kamu kuat memproses data atau tidak.

Sekarang kamu punya konteks kenapa NumPy ada. Saatnya kita belajar membuat array dan mulai bekerja dengannya.

Coba modifikasi kode di cell sebelumnya: ganti `n = 10_000_000` menjadi `n = 1_000_000` (satu juta). Jalankan lagi. Apa yang terjadi dengan rasio kecepatannya? Apakah rasionya jadi lebih besar atau lebih kecil? Apa yang bisa kamu simpulkan tentang kapan NumPy paling bermanfaat?

In [ ]:
n = 1_000_000

data_list = list(range(n))
mulai = time.time()
total_list = sum(data_list)
waktu_list = time.time() - mulai

data_np = np.arange(n)
mulai = time.time()
total_np = data_np.sum()
waktu_np = time.time() - mulai

print(f"List Python : {waktu_list:.4f} detik")
print(f"NumPy array : {waktu_np:.4f} detik")
print(f"Rasio: {waktu_list/waktu_np:.1f}x")

**Mini-check refleksi**: Coba pikirkan, untuk data berukuran 100 elemen (misalnya nilai ujian 100 mahasiswa), apakah NumPy masih terasa lebih cepat dari list biasa? Coba tebak dulu, lalu kamu bisa memodifikasi `n = 100` di cell sebelumnya untuk menguji dugaanmu. Refleksinya bukan tentang angka persis, tapi tentang *kapan* sebuah optimasi benar-benar terasa dan kapan itu cuma teori.

---

# SECTION 2: NumPy — Membuat dan Mengambil Elemen Array

Kamu sudah tahu kenapa NumPy ada. Sekarang kita belajar cara membuat array dan cara mengambil elemen tertentu dari array. Ini dua keterampilan yang akan kamu pakai setiap hari.

Ada banyak cara untuk membuat array. Yang paling dasar adalah `np.array()` yang mengkonversi list Python biasa menjadi NumPy array. Cara yang sering dipakai untuk deret angka adalah `np.arange()` (mirip `range()` tapi mengembalikan array) dan `np.linspace()` (membuat N titik yang jaraknya sama dalam sebuah rentang). Untuk array berisi nol, satu, atau nilai random, ada `np.zeros()`, `np.ones()`, dan beberapa fungsi di modul `np.random`.

Bayangkan sebuah array 1D seperti sebuah rak buku satu baris. Setiap buku punya posisi: 0, 1, 2, 3, dan seterusnya. Kamu bisa ambil buku pertama dengan indeks 0, buku terakhir dengan indeks -1, atau sekelompok buku berurutan dengan *slicing*. Untuk array 2D (matriks), bayangkan sebuah rak buku dengan baris dan kolom — kamu perlu dua indeks: `arr[baris, kolom]`.

Ketik kode di bawah ini. Kita akan membuat tiga array dengan cara berbeda, lalu mencoba mengambil elemen dan potongan dari masing-masing.

In [ ]:
genap = np.array([0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
linspace = np.linspace(0, 1, 5)
acak = np.random.randint(1, 100, 8)

print(f"np.array     : {genap}")
print(f"np.linspace  : {linspace}")
print(f"np.random... : {acak}")

Perhatikan bahwa `np.linspace(0, 1, 5)` menghasilkan lima titik yang jaraknya sama persis dari 0 sampai 1: 0, 0.25, 0.5, 0.75, dan 1. Berbeda dengan `np.arange` yang mengatur jarak antar titik, `linspace` mengatur *jumlah titik* yang kamu mau dan menghitung jaraknya sendiri. Fungsi ini sangat berguna saat kamu membuat sumbu x untuk grafik — kamu mau grafik punya 100 titik di rentang -π sampai π, bukan angka-angka acak.

Sekarang mari kita ambil elemen dari array `genap`. Array ini punya 11 elemen (dari 0 sampai 20, lompat 2), dan setiap elemen punya posisi yang disebut *index*.

Coba ambil beberapa elemen dari array `genap` di cell berikutnya. Pakai notasi bracket `[]` dengan index atau rentang. Eksperimen: ambil elemen pertama, elemen terakhir, tiga elemen pertama, dan reverse seluruh array.

In [ ]:
print(f"Elemen pertama   : {genap[0]}")
print(f"Elemen terakhir  : {genap[-1]}")
print(f"Tiga pertama     : {genap[:3]}")
print(f"Reverse          : {genap[::-1]}")
print(f"Step 2           : {genap[::2]}")

Perhatikan satu hal penting: *slicing* `genap[2:5]` mengembalikan elemen dari index 2 sampai index 4 — index 5 tidak ikut. Ini sama dengan perilaku `range(2, 5)` yang berhenti di 4. Konvensi ini berlaku di seluruh Python (list, tuple, string, dan array NumPy) dan butuh sedikit waktu untuk terbiasa. Cara yang membantu: bayangkan slice `a:b` berarti "dari index `a` sampai sebelum index `b`".

Sekarang mari kita lihat array 2D. Di NumPy, *shape* memberitahu kita berapa baris dan berapa kolom. Matriks 3x4 artinya 3 baris dan 4 kolom. Untuk mengambil elemen, kamu pakai dua index: `arr[baris, kolom]`.

Coba modifikasi kode di bawah. Buat matriks 3x4 berisi angka 0 sampai 11 dengan `np.arange(12).reshape(3, 4)`. Lalu ambil: elemen baris ke-2 kolom ke-3 (ingat, index mulai dari 0), seluruh baris pertama, dan seluruh kolom terakhir.

In [ ]:
matriks = np.arange(12).reshape(3, 4)
print(f"Matriks 3x4:\n{matriks}")
print(f"\nBaris ke-2, kolom ke-3 : {matriks[1, 2]}")
print(f"Seluruh baris pertama  : {matriks[0, :]}")
print(f"Seluruh kolom terakhir : {matriks[:, -1]}")

Perhatikan bahwa `matriks[0, :]` mengambil seluruh kolom di baris 0, sedangkan `matriks[:, -1]` mengambil seluruh baris di kolom terakhir. Tanda `:` artinya "semua". Ini notasi yang akan kamu pakai berulang kali — untuk filter, untuk operasi per baris atau per kolom, untuk normalisasi data. Kalau kamu lupa urutannya, ingat saja: *row* dulu, baru *column* — sama seperti cara kita biasa membaca koordinat di peta (baris horizontal dulu, baru vertikal).

Sekarang kamu sudah bisa membuat array dan mengambil elemen. Tapi NumPy punya satu fitur yang jauh lebih menarik: kamu bisa menyaring elemen berdasarkan kondisi, tanpa loop.

Coba buat array `acak` dari cell sebelumnya (yang berisi 8 angka random 1-99). Lalu gunakan *boolean indexing* untuk menyaring elemen yang lebih besar dari 50. Bentuknya: `acak[acak > 50]`. Perhatikan bahwa `acak > 50` sendiri menghasilkan array boolean, lalu array boolean itu dipakai untuk menyaring array asli.

In [ ]:
acak = np.random.randint(1, 100, 8)
print(f"Array asli          : {acak}")
print(f"Boolean mask (> 50) : {acak > 50}")
print(f"Elemen yang > 50    : {acak[acak > 50]}")

**Mini-check refleksi**: Kalau kamu punya array 1000 nilai ujian, dan kamu ingin tahu berapa nilai yang di atas 75 — tanpa NumPy, kamu akan menulis loop. Dengan NumPy, kamu bisa langsung pakai boolean indexing. Menurut kamu, mana yang lebih mudah dibaca? Mana yang lebih cepat? Coba pikirkan satu skenario lain di mana *boolean indexing* akan sangat menghemat waktumu (misalnya data sensor yang ingin kamu filter berdasarkan ambang batas tertentu).

---

# SECTION 3: NumPy — Operasi Vektor dan Agregasi

Salah satu hal yang sering membuat pemula kaget adalah: di NumPy, kamu tidak perlu menulis loop untuk melakukan operasi pada banyak angka. Kalau kamu punya dua array dan ingin menjumlahkannya elemen per elemen, kamu tinggal tulis `a + b` — NumPy yang akan mengurus loop-nya di balik layar. Ini disebut *vectorization*.

Analogi: bayangkan kamu seorang guru yang ingin menambahkan 5 poin ke nilai 100 siswa. Dengan list Python, kamu harus menulis loop: `nilai_baru = [n + 5 for n in nilai_lama]`. Dengan NumPy, kamu cukup tulis `nilai_lama + 5` — NumPy akan otomatis "menyebarkan" angka 5 ke setiap elemen array. Operasi yang sama, tapi cara penulisannya jauh lebih ringkas.

Vectorization bukan cuma soal keringkasan kode. Yang lebih penting: NumPy menjalankan operasi ini di kode C yang teroptimasi, sehingga jauh lebih cepat dari loop Python. Untuk data kecil perbedaannya tidak terasa, tapi untuk data jutaan elemen, perbedaannya bisa 50x atau lebih.

Selain operasi per elemen, NumPy juga punya banyak fungsi *agregasi* — fungsi yang merangkum banyak angka menjadi satu angka. Contoh: `np.sum` menjumlahkan semua elemen, `np.mean` menghitung rata-rata, `np.max` mencari nilai terbesar. Ini seperti punya kalkulator statistik built-in.

Ketik kode ini. Kita akan membuat dua array, lalu melakukan operasi matematika dasar tanpa loop. Amati outputnya — semuanya bekerja elemen per elemen.

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print(f"a + b  = {a + b}")
print(f"a * b  = {a * b}")
print(f"a ** 2 = {a ** 2}")
print(f"a + 100 = {a + 100}")

Perhatikan baris `a + 100` — ini yang disebut *scalar broadcasting*. Kita menambahkan sebuah angka tunggal (skalar) ke sebuah array, dan NumPy otomatis menerapkan 100 ke setiap elemen. Tidak ada loop, tidak ada list comprehension, tidak ada error. Ini yang akan kamu pakai setiap kali ingin menggeser atau menskalakan seluruh data sekaligus.

Sekarang mari kita lihat fungsi agregasi. Misalkan kamu punya nilai ujian 10 mahasiswa.

Coba buat array `nilai` berisi 10 angka random 50-100. Lalu hitung: total nilai (`nilai.sum()`), rata-rata (`nilai.mean()`), standar deviasi (`nilai.std()`), nilai tertinggi (`nilai.max()`), dan posisi index dari nilai tertinggi (`nilai.argmax()`). Jalankan beberapa kali — kamu akan melihat angkanya berubah karena di-generate ulang setiap kali cell di-run.

In [ ]:
nilai = np.random.randint(50, 101, 10)
print(f"Nilai        : {nilai}")
print(f"Total        : {nilai.sum()}")
print(f"Rata-rata    : {nilai.mean():.2f}")
print(f"Std dev      : {nilai.std():.2f}")
print(f"Tertinggi    : {nilai.max()}")
print(f"Index tertinggi: {nilai.argmax()}")

Perhatikan satu hal penting: `nilai.argmax()` mengembalikan *index* dari nilai tertinggi, bukan nilai tertingginya sendiri. Ini cara untuk menjawab pertanyaan "mahasiswa mana yang mendapat nilai tertinggi?" tanpa harus melihat array secara manual. Kalau kamu ingin tahu nilainya, kamu bisa tulis `nilai[nilai.argmax()]` — ambil nilai pada posisi index tertinggi.

Sekarang mari kita gabungkan semuanya. Kita akan menggunakan *boolean indexing* dan *agregasi* dalam satu ekspresi untuk menjawab pertanyaan yang lebih kompleks: "berapa nilai maksimum di atas rata-rata?"

Coba modifikasi cell di bawah. Buat array 20 nilai random 0-100. Lalu: (1) filter hanya nilai di atas 70, (2) hitung rata-rata dari nilai yang tersaring, (3) hitung berapa banyak nilai yang tersaring. Eksperimen: ubah threshold 70 menjadi 50 atau 90 — amati bagaimana hasilnya berubah.

In [ ]:
nilai = np.random.randint(0, 101, 20)
di_atas_70 = nilai[nilai > 70]

print(f"Nilai asli         : {nilai}")
print(f"Nilai di atas 70   : {di_atas_70}")
print(f"Rata-rata di atas 70: {di_atas_70.mean():.2f}")
print(f"Jumlah di atas 70  : {len(di_atas_70)}")

Kalau kamu berhasil mendapatkan output untuk semua `print` di atas, kamu baru saja melakukan analisis data lengkap dalam 6 baris kode: menghasilkan data, menyaring data, lalu merangkum hasil. Tanpa NumPy, kamu butuh loop minimal 8-10 baris. Inilah kenapa vectorization adalah kebiasaan yang sangat berharga — bukan karena "keren", tapi karena kode yang kamu tulis jadi lebih pendek dan lebih mudah dipahami.

Sekarang kita sudah bisa membuat array, mengambil elemen, dan melakukan operasi agregasi. Tapi ada satu fitur NumPy yang benar-benar membedakan dia dari list biasa: kemampuan melakukan operasi antara array dengan bentuk (*shape*) yang berbeda. Namanya *broadcasting*.

**Mini-check refleksi**: Coba pikirkan satu skenario di dunia nyata (bukan pemrograman) di mana kamu ingin melakukan operasi yang sama pada banyak objek sekaligus. Misalnya, kalau kamu punya 1000 nilai ujian dan ingin mengkonversi semua dari skala 0-100 menjadi skala 0-4 (IPK), operasi apa yang kamu lakukan pada setiap nilai? Bagaimana kalau kamu harus melakukannya secara manual — apakah kamu akan menulis formula berbeda untuk setiap nilai, atau satu formula yang sama?

---

# SECTION 4: NumPy — Broadcasting untuk Operasi Multi-Dimensi

Bayangkan kamu seorang koki yang punya banyak resep. Setiap resep punya bahan yang sama (daging, bawang, garam) tapi dalam jumlah berbeda. Kalau kamu ingin menyesuaikan semua resep untuk porsi 10 orang (bukan 5), kamu tidak perlu menulis ulang setiap resep — kamu cukup kalikan semua jumlah bahan dengan 2. *Broadcasting* di NumPy bekerja persis seperti itu: kamu bisa menerapkan satu vektor (atau skalar) ke seluruh matriks tanpa menulis loop.

Aturan main broadcasting cukup sederhana: NumPy akan "menyebarkan" array yang lebih kecil ke bentuk array yang lebih besar, selama bentuk-bentuknya kompatibel. Contoh paling sederhana: array 3x3 ditambah skalar 5 — skalar 5 dianggap ada di setiap posisi array. Contoh yang lebih menarik: matriks 3x3 ditambah vektor panjang 3 — vektor itu otomatis diterapkan ke setiap baris matriks.

Di dunia machine learning, broadcasting dipakai hampir di mana-mana. Salah satu yang paling penting adalah normalisasi: kamu punya data dengan fitur dalam skala berbeda (IPK 0-4, gaji jutaan, usia 0-100), dan kamu ingin menyamakan skalanya. Caranya: kurangi rata-rata setiap kolom, lalu bagi dengan standar deviasi. Operasi ini *harus* dilakukan per kolom, dan broadcasting adalah cara paling elegan untuk menulisnya.

Ketik kode ini. Kita akan membuat matriks nilai mahasiswa (3 baris × 3 kolom: UTS, UAS, Quiz), lalu menerapkan *broadcasting* untuk normalisasi per kolom.

In [ ]:
nilai = np.array([[72, 85, 90],
                 [68, 95, 78],
                 [88, 91, 77]])
rata_rata = nilai.mean(axis=0)
print(f"Nilai asli:\n{nilai}")
print(f"\nRata-rata per kolom: {rata_rata}")
print(f"\nNilai - rata-rata (broadcasting):\n{nilai - rata_rata}")

Perhatikan parameter `axis=0` pada `mean()`. Di NumPy, `axis=0` berarti "operasi dilakukan per kolom" — untuk setiap kolom, hitung rata-rata ke bawah (melalui semua baris). Lawannya adalah `axis=1` yang berarti "per baris". Cara mengingatnya: `axis` adalah arah *di sepanjang* mana operasi dilakukan. `axis=0` artinya sumbu vertikal — jadi kita bergerak ke bawah untuk setiap kolom. Coba pikirkan ini sebagai elevator: `axis=0` adalah elevator yang turun ke bawah, `axis=1` adalah elevator yang geser ke samping.

Hasil dari `nilai - rata_rata` adalah matriks baru di mana setiap kolom sudah punya rata-rata 0. Ini adalah langkah pertama normalisasi — langkah kedua adalah membagi dengan standar deviasi. Mari kita lihat apakah kamu bisa melakukannya sendiri.

Coba modifikasi cell di bawah. Lanjutkan dari kode sebelumnya: setelah menghitung `nilai - rata_rata`, bagi hasilnya dengan `nilai.std(axis=0)` (standar deviasi per kolom). Simpan hasilnya di variabel `ternormalisasi`. Cetak matriks hasilnya dan verifikasi bahwa rata-rata per kolom sekarang mendekati 0.

In [ ]:
nilai = np.array([[72, 85, 90],
                 [68, 95, 78],
                 [88, 91, 77]])
rata_rata = nilai.mean(axis=0)
std = nilai.std(axis=0)
ternormalisasi = (nilai - rata_rata) / std
print(f"Nilai ternormalisasi:\n{ternormalisasi.round(3)}")
print(f"\nRata-rata per kolom (harusnya ~0): {ternormalisasi.mean(axis=0).round(6)}")
print(f"Std per kolom (harusnya ~1)     : {ternormalisasi.std(axis=0).round(6)}")

Selamat — kamu baru saja melakukan normalisasi z-score, salah satu teknik preprocessing data yang paling sering dipakai di machine learning. Apa yang baru kita lakukan: untuk setiap kolom, kurangi rata-ratanya dan bagi dengan standar deviasinya. Hasilnya: kolom baru punya rata-rata 0 dan standar deviasi 1. Ini penting karena banyak algoritma ML (regresi linear, KNN, neural network) bekerja lebih baik kalau fitur-fitur inputnya punya skala yang sebanding.

Broadcasting juga bisa dipakai untuk hal yang lebih sederhana, misalnya menambahkan vektor konstanta ke setiap baris.

Coba buat matriks 4x3 berisi angka 1-12 dengan `np.arange(1, 13).reshape(4, 3)`. Lalu buat vektor `[10, 20, 30]` dan tambahkan ke matriks. Amati: vektor 3-elemen otomatis diterapkan ke setiap baris matriks 4x3. Sekarang coba buat vektor kolom 4x1 dan tambahkan juga — apa bedanya dengan vektor baris?

In [ ]:
matriks = np.arange(1, 13).reshape(4, 3)
vektor_baris = np.array([10, 20, 30])
vektor_kolom = np.array([[100], [200], [300], [400]])

print(f"Matriks 4x3:\n{matriks}")
print(f"\n+ vektor baris (panjang 3):\n{matriks + vektor_baris}")
print(f"\n+ vektor kolom (tinggi 4):\n{matriks + vektor_kolom}")

Perhatikan perbedaan di output: vektor baris (panjang 3) diterapkan ke setiap baris matriks, sedangkan vektor kolom (tinggi 4) diterapkan ke setiap kolom. Broadcasting tidak hanya "menyebarkan" ke ukuran yang sama — dia juga menghormati orientasi array. Ini fitur yang sangat kuat untuk operasi per-baris atau per-kolom tanpa loop.

Sekarang kamu sudah menguasai empat pilar NumPy: kenapa array lebih cepat, cara membuat dan mengambil elemen, operasi vektor dan agregasi, serta broadcasting. Saatnya kita tinggalkan NumPy sebentar dan masuk ke dunia baru: Pandas, tempat kita bekerja dengan data tabular.

**Mini-check refleksi**: Bayangkan kamu bekerja di startup e-commerce dan punya data penjualan harian untuk 50 produk selama 365 hari. Data disimpan dalam matriks 50x365 (baris = produk, kolom = hari). Kamu ingin mengetahui untuk setiap produk, berapa penjualan rata-ratanya dan berapa fluktuasi hariannya. Tanpa NumPy kamu akan menulis loop bersarang. Dengan NumPy, kamu bisa pakai `axis=1` untuk operasi per-baris. Coba pikirkan: operasi apa yang harus kamu pakai untuk menghitung rata-rata per produk? Untuk menghitung standar deviasi per produk? Untuk menemukan hari terbaik setiap produk?

---

# SECTION 5: Pandas — DataFrame dan CSV

NumPy sangat bagus untuk angka, tapi di dunia nyata data tidak selalu berbentuk matriks angka. Sering kali data kita punya kolom bernama (bukan hanya posisi) dan baris dengan label, dan ada campuran tipe data: nama berupa string, nilai ujian berupa angka, tanggal berupa objek khusus. Untuk data seperti ini, NumPy terasa kurang pas — kamu harus menghafal index kolom, dan tidak ada cara natural untuk memberi nama pada baris.

Pandas mengisi celah itu dengan struktur data bernama *DataFrame*. Bayangkan DataFrame sebagai spreadsheet yang bisa kamu operasikan dengan Python: ada baris, ada kolom dengan nama, dan setiap kolom boleh punya tipe data sendiri. Analogi yang sering dipakai: kolom di DataFrame seperti Series (array 1D dengan label), dan banyak Series yang berbagi index yang sama menyusun DataFrame.

Cara paling umum untuk membuat DataFrame adalah dari file CSV — format teks di mana setiap baris adalah satu record dan kolom dipisahkan oleh koma. Hampir semua data di industri (data penjualan, data pelanggan, data sensor, data Kaggle) datang dalam format ini. Karena itu, fungsi `pd.read_csv()` adalah fungsi yang paling sering kamu pakai di data science.

Ketik kode ini. Kita akan membuat DataFrame kecil dari dictionary, lalu melihat strukturnya. Tujuannya: membiasakanmu dengan output DataFrame sebelum kita membaca data dari file.

In [ ]:
data = {
    'nama'    : ['Andi', 'Budi', 'Citra', 'Dewi', 'Eka'],
    'jurusan' : ['IF', 'SI', 'IF', 'TK', 'IF'],
    'ipk'     : [3.8, 3.5, 3.9, 3.2, 3.7],
    'semester': [5, 3, 7, 2, 6]
}
df = pd.DataFrame(data)
print(df)
print(f"\nShape: {df.shape} (baris, kolom)")
print(f"Kolom: {list(df.columns)}")
print(f"Tipe data: \n{df.dtypes}")

Perhatikan output `df.dtypes`: Pandas secara otomatis mendeteksi tipe data tiap kolom. Kolom `nama` dan `jurusan` dikenali sebagai `object` (yang di Pandas berarti string atau campuran tipe), dan kolom `ipk` serta `semester` dikenali sebagai angka. Ini penting karena operasi matematika hanya bisa dilakukan pada kolom numerik.

Sekarang mari kita baca data dari CSV. Di cell berikutnya, kita akan menggunakan data CSV yang sudah ada di string — tapi di dunia nyata, kamu akan menunjuk ke file path seperti `pd.read_csv('data/penjualan.csv')`.

Coba modifikasi cell di bawah. Tambahkan satu data lagi ke dictionary `data`: nama `'Fajar'`, jurusan `'SI'`, IPK `3.6`, semester `4`. Buat DataFrame baru dan cetak. Lalu coba hapus satu baris dari dictionary dan amati bagaimana shape-nya berubah.

In [ ]:
data = {
    'nama'    : ['Andi', 'Budi', 'Citra', 'Dewi', 'Eka', 'Fajar'],
    'jurusan' : ['IF', 'SI', 'IF', 'TK', 'IF', 'SI'],
    'ipk'     : [3.8, 3.5, 3.9, 3.2, 3.7, 3.6],
    'semester': [5, 3, 7, 2, 6, 4]
}
df = pd.DataFrame(data)
print(df)
print(f"\nJumlah mahasiswa: {len(df)}")
print(f"Rata-rata IPK   : {df['ipk'].mean():.2f}")

Perhatikan dua hal: pertama, kamu bisa langsung memanggil metode `mean()` pada kolom `df['ipk']` — Pandas memperlakukan kolom sebagai Series, dan Series punya banyak metode statistik. Kedua, `len(df)` mengembalikan jumlah baris — ini cara cepat untuk tahu ukuran dataset tanpa harus mencetak seluruh isinya.

Sekarang mari kita belajar empat metode wajib yang kamu pakai setiap kali membuka dataset baru: `head()`, `info()`, `describe()`, dan `value_counts()`. Keempatnya memberikan gambaran cepat tentang isi dan kualitas data.

Coba panggil `df.head(3)`, `df.info()`, `df.describe()`, dan `df['jurusan'].value_counts()` di cell berikutnya. Perhatikan outputnya satu per satu — itu semua informasi yang kamu butuhkan untuk mulai bekerja dengan dataset baru.

In [ ]:
print("=== head(3): 3 baris pertama ===")
print(df.head(3))

print("\n=== describe(): statistik kolom numerik ===")
print(df.describe())

print("\n=== value_counts(): frekuensi tiap jurusan ===")
print(df['jurusan'].value_counts())

Tiga output di atas memberikan tiga jenis informasi berbeda. `head()` memberitahu kamu seperti apa bentuk data — nama kolom, tipe nilai, dan beberapa baris contoh. `describe()` memberikan ringkasan statistik untuk kolom numerik: rata-rata, standar deviasi, nilai minimum dan maksimum, dan quartile. `value_counts()` memberitahu berapa kali setiap nilai unik muncul di kolom kategorik — ini cara cepat untuk melihat distribusi.

Kalau kamu bekerja dengan data nyata, empat metode ini adalah langkah pertama yang selalu kamu lakukan. Setelah melihat outputnya, kamu biasanya sudah punya gambaran: ada outlier? ada missing value? distribusi data masuk akal? sebelum mulai analisis serius, kamu tahu apa yang kamu hadapi.

Sekarang kita sudah bisa membuat dan membaca DataFrame. Saatnya kita belajar cara mengambil subset data yang kita mau — ini keterampilan paling penting di Pandas.

**Mini-check refleksi**: Bayangkan kamu punya dataset penjualan online dengan 50.000 baris dan kolom seperti `tanggal`, `produk`, `kategori`, `harga`, `jumlah`, `kota`. Sebelum mulai analisis, metode mana yang kamu panggil duluan dan kenapa? Informasi apa yang kamu cari dari masing-masing? Kalau kamu hanya boleh pilih satu metode untuk pertama kali, mana yang paling penting?

---

# SECTION 6: Pandas — Memilih Baris dan Kolom dengan `.loc` dan `.iloc`

DataFrame punya dua cara untuk memilih data: `.loc` (location by label) dan `.iloc` (location by integer position). Untuk pemula, ini sering jadi sumber kebingungan karena keduanya terlihat mirip. Tapi mereka menjawab pertanyaan berbeda.

Bayangkan kamu punya lemari arsip dengan map yang ditandai label (misalnya "Surat Kontrak 2024", "Invoice Januari"). `.loc` adalah cara bertanya "tunjukkan map dengan label X". `.iloc` adalah cara bertanya "tunjukkan map di rak nomor 3 dari kiri". Untuk DataFrame yang masih punya index default 0, 1, 2, ..., perbedaan ini tidak terasa. Tapi begitu kamu set index jadi kolom tertentu (misalnya `nama`), `.loc` dan `.iloc` berperilaku sangat berbeda.

Aturan praktisnya: kalau kamu tahu *nama* kolom atau index yang kamu mau, pakai `.loc`. Kalau kamu tahu *posisi* numerik (baris ke-5, kolom ke-2), pakai `.iloc`. Untuk filter berdasarkan kondisi (misalnya `ipk > 3.5`), kamu pakai *boolean indexing* tanpa `.loc` atau `.iloc`.

Ketik kode ini. Kita akan memilih kolom, baris, dan kombinasi keduanya dari DataFrame `df`. Perhatikan perbedaan antara `df['nama']` (mengembalikan Series) dan `df[['nama', 'ipk']]` (mengembalikan DataFrame).

In [ ]:
print("=== Pilih satu kolom (Series) ===")
print(df['nama'])
print(f"\nTipe: {type(df['nama'])}")

print("\n=== Pilih dua kolom (DataFrame) ===")
print(df[['nama', 'ipk']])
print(f"\nTipe: {type(df[['nama', 'ipk']])}")

Perhatikan output `Tipe:` di bagian bawah. `df['nama']` mengembalikan Series, sedangkan `df[['nama', 'ipk']]` mengembalikan DataFrame. Ini aturan konsisten di Pandas: **bracket tunggal dengan string = Series** (1D), **bracket ganda dengan list = DataFrame** (2D). Kedengarannya remeh, tapi ini sering bikin bingung di awal — kamu mungkin berharap `df['nama']` menjadi DataFrame, padahal dia Series.

Sekarang mari kita lihat `.iloc` untuk memilih berdasarkan posisi. `iloc` menggunakan index integer murni, dimulai dari 0.

Coba modifikasi cell di bawah. Pakai `df.iloc` untuk: (1) ambil satu baris (misalnya baris pertama), (2) ambil 3 baris pertama, (3) ambil semua baris tapi hanya kolom posisi 0 dan 2, (4) ambil baris 1 sampai 3 dengan kolom posisi 0 dan 1. Perhatikan bahwa `iloc` punya format `df.iloc[baris, kolom]`.

In [ ]:
print("=== Baris pertama (iloc[0]) ===")
print(df.iloc[0])

print("\n=== 3 baris pertama (iloc[0:3]) ===")
print(df.iloc[0:3])

print("\n=== Semua baris, kolom posisi 0 dan 2 ===")
print(df.iloc[:, [0, 2]])

Perhatikan satu keanehan `.iloc` yang berbeda dari list biasa: `df.iloc[0:3]` mengambil baris 0, 1, 2 (tiga baris) — index 3 tidak ikut. Ini sesuai dengan konvensi slicing di Python dan NumPy. Tapi tunggu, di `.loc` (yang akan kita lihat nanti), slicing *inklusif* di kedua sisi. Ini jebakan umum — selalu ingat: `.iloc` konsisten dengan Python slicing, `.loc` inklusif di akhir.

Sekarang yang paling sering dipakai di dunia kerja: *boolean indexing*. Memilih baris berdasarkan kondisi. Bentuknya: `df[df['kolom'] > nilai]`.

Coba filter DataFrame `df` dengan beberapa kondisi: (1) mahasiswa dengan IPK di atas 3.5, (2) mahasiswa dari jurusan IF, (3) mahasiswa IF semester >= 5. Untuk kondisi majemuk, kamu butuh *parentheses* di setiap kondisi dan operator `&` (and) atau `|` (or).

In [ ]:
print("=== IPK di atas 3.5 ===")
print(df[df['ipk'] > 3.5])

print("\n=== Jurusan IF, semester >= 5 ===")
print(df[(df['jurusan'] == 'IF') & (df['semester'] >= 5)])

Perhatikan detail penting: setiap kondisi dibungkus kurung, dan di antara kondisi kamu pakai `&` (bukan `and` seperti di Python biasa). Ini karena `&` adalah operator bitwise yang bekerja elemen-per-elemen di Pandas, sedangkan `and` Python bekerja pada nilai truthiness tunggal. Pemula sering salah di sini dan dapat error `ValueError: The truth value of a Series is ambiguous`.

Sekarang gabungkan semuanya. Kita akan pakai `.loc` untuk menyaring baris DAN memilih kolom sekaligus — ini pola yang sangat sering kamu temui di kode Pandas.

Coba buat filter: mahasiswa IF semester >= 5, tapi hanya tampilkan kolom `nama` dan `ipk`. Gunakan `.loc[filter, [kolom]]` — format ini adalah salah satu yang paling sering kamu pakai di Pandas.

In [ ]:
filter = (df['jurusan'] == 'IF') & (df['semester'] >= 5)
hasil = df.loc[filter, ['nama', 'ipk']]
print(hasil)

Selamat — kamu baru saja menggunakan salah satu pola paling kuat di Pandas: filter baris dengan kondisi, lalu pilih kolom yang relevan, semuanya dalam dua baris kode. Pola `df.loc[kondisi, [kolom]]` akan kamu pakai ratusan kali. Kalau kamu lupa sintaksnya, ingat saja: `.loc` adalah "location by label", dan kamu bisa memberikan dua hal: baris (dengan kondisi atau label), lalu kolom (dengan list nama kolom).

Sekarang kamu sudah bisa memilih data dari DataFrame. Tapi memilih data saja tidak cukup — kita juga perlu mengelompokkan data dan menghitung ringkasan per kelompok. Itu topik berikutnya.

**Mini-check refleksi**: Kamu punya data penjualan 1000 transaksi dengan kolom `kota`, `produk`, `harga`, `tanggal`. Pertanyaan: "rata-rata harga per produk di kota Jakarta, diurutkan dari yang tertinggi". Berapa langkah yang kamu butuhkan? Langkah-langkahnya kira-kira apa saja? (Hint: boolean indexing, group by, sort.)

---

# SECTION 7: Pandas — Group By, Agregasi, dan Missing Values

Di dunia data, pertanyaan yang paling sering muncul adalah: "berapa rata-rata X untuk setiap grup Y?". Contoh: rata-rata IPK per jurusan, total penjualan per kota, jumlah mahasiswa per semester. Pola pikirnya selalu sama: *bagi data jadi beberapa grup berdasarkan satu kolom, lalu hitung statistik per grup*. Di Pandas, ini dilakukan dengan `groupby()`.

Analogi: bayangkan kamu punya tumpukan kartu mahasiswa. Kamu bisa memisahkan kartu-kartu itu jadi beberapa tumpukan kecil berdasarkan jurusannya, lalu menghitung rata-rata IPK di setiap tumpukan. `groupby()` melakukan hal yang sama, tapi di memori komputer, jauh lebih cepat dari yang bisa kamu lakukan secara manual.

Topik kedua yang akan kita bahas di section ini adalah *missing values* — data yang hilang atau tidak tercatat. Di dunia nyata, hampir tidak ada dataset yang benar-benar bersih. Ada baris yang kosong, ada kolom yang tidak terisi, ada data yang hilang karena sensor rusak atau responden tidak menjawab. Cara menangani missing values adalah keterampilan wajib di data science.

Ketik kode ini. Kita akan mengelompokkan DataFrame `df` berdasarkan `jurusan` dan menghitung rata-rata IPK per jurusan.

In [ ]:
print("=== Rata-rata IPK per jurusan ===")
print(df.groupby('jurusan')['ipk'].mean().round(3))

print("\n=== Jumlah mahasiswa per jurusan ===")
print(df.groupby('jurusan').size())

Perhatikan pola `df.groupby('jurusan')['ipk'].mean()`. Ini dibaca seperti kalimat bahasa Inggris: *"group by jurusan, ambil kolom ipk, hitung mean"*. Pandas mengimplementasikan ini dalam tiga langkah: (1) pisah data jadi grup berdasarkan nilai unik di kolom `jurusan`, (2) untuk setiap grup, ambil kolom `ipk`, (3) hitung mean per grup. Hasilnya adalah Series baru yang indexnya adalah nilai unik jurusan.

Kamu juga bisa menghitung beberapa agregasi sekaligus dengan `.agg()`.

Coba modifikasi cell di bawah. Pakai `df.groupby('jurusan').agg({'ipk': ['mean', 'min', 'max']})` untuk menghitung tiga statistik sekaligus. Perhatikan outputnya: hasilnya adalah DataFrame dengan kolom multi-level.

In [ ]:
print("=== Multiple aggregations ===")
print(df.groupby('jurusan').agg({
    'ipk'     : ['mean', 'min', 'max'],
    'semester': 'mean'
}).round(2))

Perhatikan struktur output: kolom-kolomnya sekarang punya dua level — di level pertama adalah nama kolom asli (`ipk`, `semester`), di level kedua adalah nama fungsi agregasi (`mean`, `min`, `max`). Struktur seperti ini disebut *MultiIndex* di Pandas, dan kelihatannya rumit, tapi sebenarnya hanya menumpuk informasi: "kolom mana yang dihitung, dengan fungsi apa".

Sekarang mari kita masuk ke missing values. Pertama, kita buat DataFrame yang punya missing values, lalu belajar cara mendeteksi dan menanganinya.

Coba buat DataFrame baru dengan 4-5 baris, di mana beberapa nilai `ipk` dan `nilai_uts` adalah `None` (Python's null). Lalu panggil `df.isnull().sum()` untuk mendeteksi missing values per kolom. Amati output: baris True menunjukkan posisi missing.

In [ ]:
data_nan = {
    'nama'     : ['Andi', 'Budi', 'Citra', 'Dewi', 'Eka'],
    'ipk'      : [3.8, None, 3.9, 3.2, None],
    'nilai_uts': [85, 78, None, 70, 82]
}
df_nan = pd.DataFrame(data_nan)
print("DataFrame dengan missing values:")
print(df_nan)
print(f"\nJumlah missing per kolom:\n{df_nan.isnull().sum()}")

Perhatikan bahwa `isnull()` mengembalikan DataFrame boolean, lalu `.sum()` menghitung jumlah `True` per kolom. Ini pola yang sangat umum di Pandas: *metode boolean dulu (menghasilkan True/False), lalu agregasi (menghitung atau memfilter)*. Pola ini juga berlaku untuk metode lain seperti `isna()` (sinonim untuk `isnull()`), `notnull()`, dan seterusnya.

Ada dua strategi utama untuk menangani missing values. Pertama, *drop* — hapus baris atau kolom yang punya missing. Kedua, *fill* — isi missing dengan nilai tertentu. Mari kita lihat keduanya.

Coba dua strategi di cell berikutnya: (1) `df_nan.dropna()` untuk hapus baris yang ada missing, (2) isi kolom `ipk` dengan mean dan kolom `nilai_uts` dengan median. Untuk fillna, sintaksnya: `df_nan['kolom'].fillna(nilai_pengganti)`. Amati perbedaannya: dropna mengubah ukuran data, fillna mempertahankan ukuran.

In [ ]:
print("=== dropna (hapus baris dengan missing) ===")
print(df_nan.dropna())

print("\n=== fillna (isi dengan mean/median) ===")
df_filled = df_nan.copy()
df_filled['ipk'] = df_filled['ipk'].fillna(df_filled['ipk'].mean())
df_filled['nilai_uts'] = df_filled['nilai_uts'].fillna(df_filled['nilai_uts'].median())
print(df_filled)
print(f"\nMissing setelah fillna: {df_filled.isnull().sum().sum()}")

Perhatikan baris terakhir: setelah fillna, `df_filled.isnull().sum().sum()` mengembalikan 0. Mengapa ada dua `.sum()`? Yang pertama menjumlahkan `True` per kolom (menghasilkan Series), yang kedua menjumlahkan seluruh angka di Series itu (menghasilkan skalar). Ini cara cepat untuk cek apakah masih ada missing values di seluruh DataFrame.

Strategi mana yang lebih baik? Jawabannya: tergantung situasi. Kalau missing values cuma sedikit (misalnya < 5% data), `dropna()` aman. Kalau missing values banyak, `dropna()` akan menghilangkan terlalu banyak data, dan `fillna()` lebih baik. Untuk data yang punya urutan (time series), mengisi dengan nilai sebelumnya (forward fill) atau nilai sesudahnya (backward fill) sering lebih masuk akal daripada mean.

Sekarang kamu sudah punya bekal NumPy dan Pandas yang cukup solid. Saatnya kita gunakan semuanya di mini project EDA.

**Mini-check refleksi**: Kamu punya dataset gaji karyawan dengan kolom `gaji`, `tahun_masuk`, `departemen`, dan 5% data `gaji` hilang. Pertanyaan: apakah lebih baik dropna atau fillna? Kalau fillna, apa nilai yang masuk akal? Apakah mean dari seluruh data? mean per departemen? median? Coba pikirkan trade-off-nya — apa risiko dari masing-masing strategi?

---

# SECTION 8: Mini Project — EDA pada Dataset Mahasiswa

Sekarang kita gabungkan semua yang sudah kita pelajari untuk melakukan Exploratory Data Analysis (EDA) lengkap. EDA adalah proses sistematis untuk memahami dataset: cek dimensi, cek missing values, hitung statistik deskriptif, lihat korelasi, dan visualisasi distribusi. Ini langkah pertama sebelum masuk ke machine learning — kamu harus tahu data kamu dulu sebelum memutuskan model apa yang akan dipakai.

Di project ini kita akan bekerja dengan dataset sintetis 100 mahasiswa. Kita akan generate data secara random, lalu melakukan EDA langkah demi langkah. Tujuannya bukan untuk dapat insight bisnis (data ini memang sintetis), tapi untuk melatih alur kerja EDA yang akan kamu pakai di proyek nyata.

Alur EDA yang akan kita ikuti: (1) Load dan cek dimensi, (2) cek tipe data dan missing values, (3) hitung statistik deskriptif, (4) buat korelasi heatmap, (5) visualisasi distribusi, (6) bandingkan antar grup, (7) tarik insight. Ikuti setiap langkah, amati output, dan pikirkan: "kalau ini data sungguhan, informasi apa yang bisa saya dapat?"

Ketik kode ini. Kita akan generate dataset 100 mahasiswa dengan informasi nama, jurusan, semester, IPK, dan nilai ujian. Pakai `np.random.seed` supaya hasilnya reproducible — kalau kamu jalankan ulang, datanya akan persis sama.

In [ ]:
np.random.seed(100)

nama_depan = ['Andi', 'Budi', 'Citra', 'Dewi', 'Eka', 'Fajar', 'Gita',
              'Hadi', 'Intan', 'Joko', 'Kiki', 'Lina', 'Mira', 'Niko']
nama_belakang = ['Pratama', 'Wijaya', 'Kusuma', 'Nugroho', 'Santoso']
jurusan_list = ['IF', 'SI', 'TK', 'DKV']

data = {
    'nama'     : [np.random.choice(nama_depan) + ' ' + np.random.choice(nama_belakang) for _ in range(100)],
    'jurusan'  : np.random.choice(jurusan_list, 100),
    'semester' : np.random.randint(1, 9, 100),
    'ipk'      : np.clip(np.random.normal(3.4, 0.4, 100), 2.0, 4.0).round(2),
    'nilai_uts': np.clip(np.random.normal(72, 12, 100), 40, 100).astype(int),
    'nilai_uas': np.clip(np.random.normal(75, 10, 100), 40, 100).astype(int),
}
df = pd.DataFrame(data)
print(f"Dataset berhasil dibuat. Shape: {df.shape}")
print(df.head())

Perhatikan bahwa `np.clip` memastikan nilai IPK ada di rentang 2.0 sampai 4.0 (tidak mungkin lebih dari 4 atau kurang dari 2 untuk IPK realistis). Ini teknik yang sering dipakai saat generate data sintetis: kamu buat data dengan distribusi tertentu, lalu "kliping" ke rentang yang masuk akal. Sekarang mari kita mulai EDA.

**Step 1 dan 2**: Cek dimensi dan missing values. Coba panggil `df.info()` di cell berikutnya — itu satu fungsi yang memberikan semua informasi: jumlah baris, jumlah kolom, tipe data tiap kolom, dan jumlah nilai non-null. Kalau jumlah non-null kurang dari total baris, berarti ada missing.

In [ ]:
print("=== Dimensi ===")
print(f"Baris: {df.shape[0]}, Kolom: {df.shape[1]}")
print(f"\nKolom: {list(df.columns)}")

print("\n=== Missing values per kolom ===")
print(df.isnull().sum())

print("\n=== Tipe data ===")
print(df.dtypes)

**Step 3**: Statistik deskriptif. `df.describe()` memberikan ringkasan otomatis untuk kolom numerik: count, mean, std, min, Q1, median (Q2), Q3, max. Ini cara tercepat untuk melihat distribusi data dan mendeteksi outlier (misalnya IPK yang lebih dari 4 atau kurang dari 2).

Coba panggil `df.describe()` dan `df['jurusan'].value_counts()` di cell berikutnya. Yang pertama untuk statistik numerik, yang kedua untuk distribusi kategori.

In [ ]:
print("=== Statistik kolom numerik ===")
print(df.describe().round(2))

print("\n=== Distribusi mahasiswa per jurusan ===")
print(df['jurusan'].value_counts())

print("\n=== Rata-rata IPK per jurusan ===")
print(df.groupby('jurusan')['ipk'].mean().round(3))

**Step 4**: Korelasi. Kita ingin tahu apakah IPK berkorelasi dengan nilai ujian. Korelasi diukur dengan angka antara -1 dan 1: makin dekat ke 1, makin kuat hubungan linear positif; makin dekat ke -1, makin kuat hubungan linear negatif; dekat 0 berarti tidak ada hubungan linear.

Coba hitung korelasi antar kolom numerik (`ipk`, `nilai_uts`, `nilai_uas`, `semester`) dengan `df[['ipk', 'nilai_uts', 'nilai_uas', 'semester']].corr()`. Cetak hasilnya. Amati: kolom mana yang paling berkorelasi dengan IPK?

In [ ]:
korelasi = df[['ipk', 'nilai_uts', 'nilai_uas', 'semester']].corr().round(3)
print(korelasi)

**Step 5**: Visualisasi distribusi. Sekarang kita masuk ke bagian visual. Untuk EDA, visualisasi jauh lebih cepat dibaca daripada tabel angka. Kita akan pakai Matplotlib untuk beberapa plot dasar. Di cell berikutnya, kita akan membuat histogram IPK dengan garis rata-rata.

Coba buat histogram IPK dengan matplotlib. Sintaksnya: `plt.hist(df['ipk'], bins=15, color='steelblue', edgecolor='white')`. Lalu tambahkan `plt.title('Distribusi IPK')`, `plt.xlabel('IPK')`, `plt.ylabel('Frekuensi')`, dan `plt.show()`.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df['ipk'], bins=15, color='steelblue', edgecolor='white')
plt.axvline(df['ipk'].mean(), color='red', linestyle='--', linewidth=2,
            label=f"Mean = {df['ipk'].mean():.2f}")
plt.title('Distribusi IPK', fontsize=14, fontweight='bold')
plt.xlabel('IPK')
plt.ylabel('Frekuensi')
plt.legend()
plt.tight_layout()
plt.show()

Perhatikan bahwa histogram di sini bukan sekadar visual — kamu bisa membaca beberapa insight langsung dari bentuknya. Distribusi IPK terlihat agak condong ke kiri (skewed left): ada lebih banyak mahasiswa dengan IPK tinggi, dan sedikit mahasiswa dengan IPK rendah. Ini umum di data akademik.

**Step 6**: Bandingkan antar grup dengan box plot. Box plot adalah cara cepat untuk membandingkan distribusi antar kategori. Format di matplotlib: `plt.boxplot([data_per_kategori], labels=[nama_kategori])`.

Coba buat box plot IPK per jurusan. Pertama, kumpulkan data per jurusan: `data_per_jurusan = [df[df['jurusan'] == j]['ipk'] for j in df['jurusan'].unique()]`. Lalu buat box plot dengan `plt.boxplot(data_per_jurusan, labels=df['jurusan'].unique())`.

In [ ]:
data_per_jurusan = [df[df['jurusan'] == j]['ipk'] for j in sorted(df['jurusan'].unique())]
label_jurusan = sorted(df['jurusan'].unique())

plt.figure(figsize=(8, 5))
plt.boxplot(data_per_jurusan, labels=label_jurusan, patch_artist=True,
            boxprops=dict(facecolor='lightblue'))
plt.title('Distribusi IPK per Jurusan', fontsize=14, fontweight='bold')
plt.xlabel('Jurusan')
plt.ylabel('IPK')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Step 7**: Tarik insight. EDA bukan hanya tentang menghasilkan plot dan tabel — itu tentang mengubah data menjadi cerita. Di cell berikutnya, kita akan gabungkan beberapa analisis untuk menjawab pertanyaan bisnis: jurusan mana yang punya IPK rata-rata tertinggi? Apakah IPK berkorelasi dengan nilai ujian? Apakah semester mempengaruhi IPK?

Coba buat scatter plot IPK vs nilai_uts. Sintaksnya: `plt.scatter(df['ipk'], df['nilai_uts'], alpha=0.6)`. Tambahkan label sumbu dan judul. Amati: ada pola naik (kalau IPK naik, UTS juga naik)? Ini akan menjawab pertanyaan "apakah IPK dan UTS berkorelasi?" secara visual.

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df['ipk'], df['nilai_uts'], alpha=0.6, color='steelblue')
plt.title('IPK vs Nilai UTS', fontsize=14, fontweight='bold')
plt.xlabel('IPK')
plt.ylabel('Nilai UTS')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Korelasi IPK-UTS: {df['ipk'].corr(df['nilai_uts']):.3f}")

Perhatikan angka korelasi yang dicetak di bawah plot. Kalau angkanya di atas 0.5, ada korelasi positif yang kuat — mahasiswa dengan IPK tinggi cenderung punya nilai UTS yang juga tinggi. Kalau di bawah 0.3, korelasinya lemah dan kamu tidak bisa bilang IPK memprediksi UTS (atau sebaliknya).

Selamat — kamu sudah menyelesaikan mini project EDA! Kamu sudah melalui langkah-langkah paling penting: load data, cek kualitas, hitung statistik, visualisasi, dan tarik insight. Ini pola yang akan kamu ulangi di hampir semua proyek data science.

Sebagai penutup, mari kita rangkum apa yang sudah kita pelajari di chapter ini.

Coba buat satu dashboard ringkas yang menampilkan 4 visualisasi dalam 1 figure: (1) histogram IPK, (2) bar chart jumlah mahasiswa per jurusan, (3) box plot IPK per jurusan, (4) scatter plot IPK vs nilai_uts. Pakai `plt.subplots(2, 2)` untuk grid 2x2. Ini adalah bentuk dashboard sederhana yang umum dipakai di dunia kerja.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].hist(df['ipk'], bins=15, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribusi IPK', fontweight='bold')
axes[0, 0].set_xlabel('IPK')
axes[0, 0].set_ylabel('Frekuensi')

df['jurusan'].value_counts().plot(kind='bar', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Mahasiswa per Jurusan', fontweight='bold')
axes[0, 1].set_xlabel('Jurusan')
axes[0, 1].set_ylabel('Jumlah')

data_jur = [df[df['jurusan'] == j]['ipk'] for j in sorted(df['jurusan'].unique())]
axes[1, 0].boxplot(data_jur, labels=sorted(df['jurusan'].unique()), patch_artist=True,
                   boxprops=dict(facecolor='lightblue'))
axes[1, 0].set_title('IPK per Jurusan', fontweight='bold')
axes[1, 0].set_xlabel('Jurusan')
axes[1, 0].set_ylabel('IPK')

axes[1, 1].scatter(df['ipk'], df['nilai_uts'], alpha=0.6, color='steelblue')
axes[1, 1].set_title('IPK vs Nilai UTS', fontweight='bold')
axes[1, 1].set_xlabel('IPK')
axes[1, 1].set_ylabel('Nilai UTS')

plt.suptitle('Dashboard EDA — Dataset Mahasiswa', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n=== Insight utama ===")
print(f"1. Jurusan dengan rata-rata IPK tertinggi: "
      f"{df.groupby('jurusan')['ipk'].mean().idxmax()} "
      f"({df.groupby('jurusan')['ipk'].mean().max():.3f})")
print(f"2. Korelasi IPK-UTS: {df['ipk'].corr(df['nilai_uts']):.3f}")
print(f"3. Korelasi IPK-UAS: {df['ipk'].corr(df['nilai_uas']):.3f}")
print(f"4. Korelasi IPK-Semester: {df['ipk'].corr(df['semester']):.3f}")

Selamat — kamu baru saja menyelesaikan chapter 2!

**Yang sudah kita pelajari:**

Di dunia NumPy, kamu sekarang paham kenapa array lebih cepat dari list (data contiguous di memori, operasi di kode C), cara membuat array dengan `np.array`, `np.arange`, `np.linspace`, dan `np.random`, cara mengambil elemen dengan indexing dan slicing, operasi vektor dan agregasi (`sum`, `mean`, `std`, `max`), boolean indexing untuk menyaring data berdasarkan kondisi, dan broadcasting untuk operasi antara array dengan bentuk berbeda.

Di dunia Pandas, kamu sekarang paham apa itu DataFrame (struktur tabular 2D dengan kolom bernama), cara membuat DataFrame dari dictionary dan membaca dari CSV, empat metode wajib (`head`, `info`, `describe`, `value_counts`), cara memilih data dengan `.loc`, `.iloc`, dan boolean indexing, group by untuk agregasi, dan cara menangani missing values dengan `dropna` dan `fillna`.

Kamu juga sudah melakukan EDA lengkap pada dataset sintetis — pola yang akan kamu ulangi di hampir semua proyek data science.

**Untuk selanjutnya**, chapter 03 akan masuk ke Machine Learning Fundamental. NumPy dan Pandas yang baru kamu pelajari akan jadi bekal utama: NumPy untuk operasi matematika di balik algoritma ML, dan Pandas untuk menyiapkan data sebelum dimasukkan ke model. Sekarang setelah fondasi analisis data sudah kuat, kamu siap untuk mulai melatih model pertamamu.